# Voiceprint — Préparation du corpus Common Voice 27.0 (metric learning v2)

**Où on est :** jour 6, préparation du corpus d'entraînement du metric learning v2.

**Ce que fait ce notebook :**
1. Télécharge Common Voice 27.0 français (28,55 Go) **sur Colab**, pas sur ton PC.
2. Exclut tous les locuteurs de validation et de test de cv26 (anti-fuite, vérification automatique).
3. Choisit environ 1 000 locuteurs, 12 clips chacun, **équilibrés par hauteur de voix (F0)**.
4. Calcule les empreintes ECAPA sur le GPU : 6 versions par clip (2 propres, ambiance 20/10 dB, brouhaha 20/10 dB), extraits de 3 s.
5. Sauvegarde dans ton Drive : l'audio choisi (zip), la liste, les empreintes. Télécharge aussi le petit corpus de **parole spontanée** (évaluation seulement).

**Avant de lancer :**
- Menu **Exécution > Modifier le type d'exécution > GPU T4**.
- Icône 🔑 (Secrets) à gauche : ajoute `MDC_API_KEY` = ta clé, et active **Accès au notebook**.
- Dépose `exclusions_cv26.csv` dans **Mon Drive/voiceprint/** (fichier créé sur ton PC par `python src/exporter_exclusions.py`).
- Conditions des deux datasets acceptées sur le site Mozilla Data Collective.

Lance les cellules **une par une, dans l'ordre**. Chaque cellule affiche ce qu'il faut me renvoyer.

In [ ]:
# 1) Réglages et vérifications de la machine
import os, shutil, subprocess

DOSSIER_DRIVE = "/content/drive/MyDrive/voiceprint"   # dossier dans ton Google Drive
TRAVAIL = "/content/travail"                           # disque temporaire de Colab (effacé en fin de session)

ID_LU = "cmu5kub4600pkmh078n7n5ogl"         # Common Voice Scripted Speech 27.0 - French
ID_SPONTANE = "cmu5z135s00ginq072mliz1as"   # Common Voice Spontaneous Speech 5.0 - French

N_LOCUTEURS = 1000         # locuteurs gardés pour l'entraînement
N_CANDIDATS = 3000         # locuteurs tirés au hasard avant l'équilibrage par F0
CLIPS_A_EXTRAIRE = 15      # clips extraits par candidat (marge pour les clips trop courts)
CLIPS_PAR_LOCUTEUR = 12    # plafond de clips gardés par locuteur
CLIPS_MIN = 8              # en dessous, le locuteur est retiré
SEUIL_F0_AIGU = 165.0      # Hz : sépare voix graves / voix aiguës pour l'équilibrage
DUREE_EXTRAIT = 3.0        # s : même durée que l'enrôlement visé le jour J
DUREE_MAX_PAROLE = 8.0     # s : on garde au plus 8 s de parole par clip (mémoire)
GRAINE = 42                # pour que les tirages au hasard soient reproductibles

os.makedirs(TRAVAIL, exist_ok=True)

import torch
gpu = torch.cuda.is_available()
print("GPU :", torch.cuda.get_device_name(0) if gpu else "AUCUN")
if not gpu:
    print("⚠️  Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4, puis relance cette cellule.")
libre = shutil.disk_usage("/content").free / 1e9
print(f"Disque libre sur Colab : {libre:.0f} Go (il en faut au moins 35)")

In [ ]:
# 2) Connexion à Google Drive et vérification du fichier d'exclusions
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DOSSIER_DRIVE, exist_ok=True)

FICHIER_EXCLUSIONS = f"{DOSSIER_DRIVE}/exclusions_cv26.csv"
if not os.path.exists(FICHIER_EXCLUSIONS):
    raise FileNotFoundError(
        "exclusions_cv26.csv introuvable dans Mon Drive/voiceprint/. "
        "Crée-le sur ton PC avec : python src/exporter_exclusions.py, puis dépose-le dans ce dossier.")
import pandas as pd
excl = pd.read_csv(FICHIER_EXCLUSIONS, dtype=str)
print("Exclusions chargées :", excl["type"].value_counts().to_dict())

In [ ]:
# 3) Installation des bibliothèques (même version de SpeechBrain que sur ton PC)
!pip install -q "speechbrain==1.1.2" praat-parselmouth
# SpeechBrain n'est importé qu'à la cellule 11 (évite un conflit avec le calcul en parallèle des cellules 9-10)
from importlib.metadata import version
import librosa, parselmouth
print("speechbrain", version("speechbrain"), "| librosa", librosa.__version__, "| torch", torch.__version__)

In [ ]:
# 4) Fonction de téléchargement (API Mozilla Data Collective, reprise automatique si coupure)
import requests, time
from urllib.parse import urlparse
from google.colab import userdata

CLE = userdata.get("MDC_API_KEY")   # lue dans les Secrets, jamais écrite dans le code
# Nouvelle adresse du site. L'ancienne redirige, et une redirection transforme POST en GET -> erreur 405.
API = "https://mozilladatacollective.com/api"

def telecharger(id_dataset, destination):
    if os.path.exists(destination + ".ok"):
        print("Déjà téléchargé :", destination); return
    r = requests.post(f"{API}/datasets/{id_dataset}/download",
                      headers={"Authorization": f"Bearer {CLE}"}, timeout=60, allow_redirects=False)
    if r.is_redirect:   # si le site a encore changé d'adresse : on refait un POST à la nouvelle adresse
        r = requests.post(r.headers["Location"], headers={"Authorization": f"Bearer {CLE}"}, timeout=60)
    if r.status_code == 401:
        raise RuntimeError("Clé API refusée : vérifie le secret MDC_API_KEY (et 'Accès au notebook' activé).")
    if r.status_code == 403:
        raise RuntimeError("Accès refusé : accepte les conditions du dataset sur le site (bouton Download).")
    if r.status_code == 429:
        raise RuntimeError("Limite de téléchargements atteinte pour aujourd'hui : réessaie demain.")
    r.raise_for_status()
    info = r.json()
    url = info.get("downloadUrl") or info.get("url")
    if not url:
        raise RuntimeError(f"Pas d'URL dans la réponse. Clés reçues : {list(info)}")

    entetes = {}
    if urlparse(url).netloc in ("mozilladatacollective.com", "datacollective.mozillafoundation.org"):
        entetes["Authorization"] = f"Bearer {CLE}"   # URL du site lui-même (pas du stockage) : il faut la clé
    deja = os.path.getsize(destination) if os.path.exists(destination) else 0
    if deja:
        entetes["Range"] = f"bytes={deja}-"
    with requests.get(url, headers=entetes, stream=True, timeout=120) as rep:
        if deja and rep.status_code == 200:   # le serveur ignore la reprise : on recommence
            deja = 0
        rep.raise_for_status()
        total = int(rep.headers.get("Content-Length", 0)) + deja
        debut, ecrit, prochain = time.time(), deja, deja + 2e9
        with open(destination, "ab" if deja else "wb") as f:
            for morceau in rep.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(morceau); ecrit += len(morceau)
                if ecrit >= prochain:
                    vitesse = (ecrit - deja) / 1e6 / (time.time() - debut)
                    print(f"  {ecrit/1e9:.1f} / {total/1e9:.1f} Go  ({vitesse:.0f} Mo/s)"); prochain += 2e9
    if total and os.path.getsize(destination) != total:
        raise RuntimeError("Téléchargement incomplet : relance la cellule, il reprendra où il s'est arrêté.")
    open(destination + ".ok", "w").close()
    print(f"OK : {destination} ({os.path.getsize(destination)/1e9:.2f} Go)")

print("Clé trouvée :", "oui" if CLE else "NON")

In [ ]:
# 5) Téléchargement de Common Voice 27.0 français (environ 5 à 20 min)
ARCHIVE = f"{TRAVAIL}/cv27_fr.tar.gz"
telecharger(ID_LU, ARCHIVE)

In [ ]:
# 6) Passage 1 : on extrait seulement les tableaux (.tsv), pas les 870 000 audios (environ 5 min)
import glob, tarfile
EXTRAIT = f"{TRAVAIL}/extrait"
os.makedirs(EXTRAIT, exist_ok=True)

with tarfile.open(ARCHIVE, "r|gz") as t:          # premier nom de l'archive : sert à connaître le préfixe
    PREMIER_MEMBRE = next(iter(t)).name
PREFIXE_POINT = "./" if PREMIER_MEMBRE.startswith("./") else ""

if not glob.glob(f"{EXTRAIT}/**/validated.tsv", recursive=True):
    subprocess.run(["tar", "-xzf", ARCHIVE, "-C", EXTRAIT, "--wildcards", "*.tsv"], check=True)

tsvs = sorted(glob.glob(f"{EXTRAIT}/**/*.tsv", recursive=True))
for f in tsvs:
    print(f"{os.path.getsize(f)/1e6:8.1f} Mo  {os.path.relpath(f, EXTRAIT)}")
VALIDATED = [f for f in tsvs if os.path.basename(f) == "validated.tsv"][0]
DUREES = [f for f in tsvs if os.path.basename(f) == "clip_durations.tsv"]
DOSSIER_CLIPS_ARCHIVE = PREFIXE_POINT + os.path.relpath(os.path.dirname(VALIDATED), EXTRAIT) + "/clips/"
print("\nClips dans l'archive sous :", DOSSIER_CLIPS_ARCHIVE)

In [ ]:
# 7) Sélection des candidats + exclusion des locuteurs de validation/test cv26 (anti-fuite)
import csv, numpy as np
rng = np.random.default_rng(GRAINE)

cols = lambda c: c in ("client_id", "path", "sentence", "sentence_id")
val = pd.read_csv(VALIDATED, sep="\t", quoting=csv.QUOTE_NONE, dtype=str, usecols=cols, on_bad_lines="skip")
val = val.dropna(subset=["client_id", "path"])
val["clip"] = val["path"].str.replace(r"\.mp3$", "", regex=True)
print(f"Clips validés : {len(val):,} | locuteurs : {val['client_id'].nunique():,}")

# Clips trop courts écartés (il faut au moins 3 s de parole après retrait des silences)
if DUREES:
    d = pd.read_csv(DUREES[0], sep="\t", dtype=str)
    d.columns = ["clip_d", "ms"]
    d["clip"] = d["clip_d"].str.replace(r"\.mp3$", "", regex=True)
    val = val.merge(d[["clip", "ms"]], on="clip", how="left")
    val = val[pd.to_numeric(val["ms"], errors="coerce").fillna(0) >= 3800]
    print(f"Après retrait des clips < 3,8 s : {len(val):,}")

# --- Anti-fuite : double vérification (client_id ET noms de clips) ---
ids_excl = set(excl.loc[excl["type"] == "client_id", "valeur"])
clips_excl = set(excl.loc[excl["type"] == "clip", "valeur"])
tous = pd.read_csv(VALIDATED, sep="\t", quoting=csv.QUOTE_NONE, dtype=str,
                   usecols=["client_id", "path"], on_bad_lines="skip")
tous["clip"] = tous["path"].str.replace(r"\.mp3$", "", regex=True)
ids_trouves = ids_excl & set(tous["client_id"])
lignes_clips = tous[tous["clip"].isin(clips_excl)]
ids_par_clips = set(lignes_clips["client_id"])
EXCLUS = ids_trouves | ids_par_clips
print(f"\nAnti-fuite : {len(ids_trouves)}/{len(ids_excl)} client_id cv26 retrouvés ; "
      f"{len(lignes_clips)}/{len(clips_excl)} clips cv26 retrouvés -> {len(ids_par_clips)} locuteurs")
print(f"Locuteurs exclus au total : {len(EXCLUS)}")
if not ids_trouves and lignes_clips.empty:
    raise RuntimeError("Aucun locuteur cv26 retrouvé : impossible de garantir l'absence de fuite. Envoie-moi ce message.")

pool = val[~val["client_id"].isin(EXCLUS)]
cle_phrase = "sentence_id" if "sentence_id" in pool.columns else "sentence"
pool = pool.drop_duplicates(subset=["client_id", cle_phrase])        # phrases différentes seulement
compte = pool["client_id"].value_counts()
eligibles = compte[compte >= CLIPS_PAR_LOCUTEUR].index.to_numpy()
print(f"\nLocuteurs avec au moins {CLIPS_PAR_LOCUTEUR} clips (hors exclus) : {len(eligibles):,}")

candidats = rng.choice(eligibles, size=min(N_CANDIDATS, len(eligibles)), replace=False)
morceaux = []
for cid, groupe in pool[pool["client_id"].isin(set(candidats))].groupby("client_id"):
    morceaux.append(groupe.sample(n=min(CLIPS_A_EXTRAIRE, len(groupe)), random_state=int(rng.integers(1e9))))
choix = pd.concat(morceaux)[["client_id", "path", "clip"]].reset_index(drop=True)
assert not set(choix["client_id"]) & EXCLUS, "FUITE : un locuteur exclu est dans la sélection !"
print(f"Candidats : {choix['client_id'].nunique():,} locuteurs, {len(choix):,} clips à extraire")
print("Vérification anti-fuite : intersection vide ✅")

LISTE = f"{TRAVAIL}/liste_clips.txt"
with open(LISTE, "w") as f:
    f.write("\n".join(DOSSIER_CLIPS_ARCHIVE + p for p in choix["path"]) + "\n")

In [ ]:
# 8) Passage 2 : extraction des seuls clips choisis (environ 5 min)
CLIPS_LOCAUX = os.path.join(EXTRAIT, DOSSIER_CLIPS_ARCHIVE.removeprefix("./"))
subprocess.run(["tar", "-xzf", ARCHIVE, "-C", EXTRAIT, "-T", LISTE], check=True)
choix["fichier"] = CLIPS_LOCAUX + choix["path"]
presents = choix["fichier"].map(os.path.exists)
print(f"Clips extraits : {presents.sum():,} / {len(choix):,}")
choix = choix[presents].reset_index(drop=True)

In [ ]:
# 9) Hauteur de voix (F0) des candidats, puis équilibrage voix graves / voix aiguës
from multiprocessing import Pool

def charger_parole(fichier):
    # Même nettoyage que la démo : 16 kHz mono, silences et pauses retirés (top_db=30)
    y, _ = librosa.load(fichier, sr=16000, mono=True)
    morceaux = librosa.effects.split(y, top_db=30)
    if len(morceaux) == 0:
        return np.zeros(0, dtype=np.float32)
    parole = np.concatenate([y[a:b] for a, b in morceaux]).astype(np.float32)
    return parole[: int(DUREE_MAX_PAROLE * 16000)]

def f0_clip(fichier):
    try:
        parole = charger_parole(fichier)
        if len(parole) < 16000:
            return np.zeros(0)
        hauteur = parselmouth.Sound(parole.astype(np.float64), 16000).to_pitch(
            time_step=0.01, pitch_floor=60, pitch_ceiling=500)
        f0 = hauteur.selected_array["frequency"]
        return f0[f0 > 0]
    except Exception:
        return np.zeros(0)

deux = choix.groupby("client_id").head(2)          # 2 clips par candidat suffisent pour la F0
with Pool(os.cpu_count()) as p:
    valeurs = p.map(f0_clip, deux["fichier"].tolist(), chunksize=16)
f0_par_loc = {}
for cid, v in zip(deux["client_id"], valeurs):
    f0_par_loc.setdefault(cid, []).append(v)
f0_med = pd.Series({cid: np.median(np.concatenate(v)) if sum(len(x) for x in v) > 20 else np.nan
                    for cid, v in f0_par_loc.items()}).dropna()

aigus = f0_med[f0_med >= SEUIL_F0_AIGU].index.to_numpy()
graves = f0_med[f0_med < SEUIL_F0_AIGU].index.to_numpy()
print(f"Candidats mesurés : {len(f0_med)} | graves (< {SEUIL_F0_AIGU:.0f} Hz) : {len(graves)} | aigus : {len(aigus)}")
n_aigus = min(len(aigus), N_LOCUTEURS // 2)
n_graves = min(len(graves), N_LOCUTEURS - n_aigus)
n_aigus = min(len(aigus), N_LOCUTEURS - n_graves)    # complète avec des aigus s'il manque des graves
garde = np.concatenate([rng.choice(graves, n_graves, replace=False), rng.choice(aigus, n_aigus, replace=False)])
print(f"Gardés : {len(garde)} locuteurs ({n_graves} graves, {n_aigus} aigus)")

In [ ]:
# 10) Chargement de la parole des locuteurs gardés, contrôle des durées, sauvegarde sur Drive
sel = choix[choix["client_id"].isin(set(garde))].reset_index(drop=True)
with Pool(os.cpu_count()) as p:
    paroles = p.map(charger_parole, sel["fichier"].tolist(), chunksize=16)
sel["duree_parole"] = [len(x) / 16000 for x in paroles]
ok = sel["duree_parole"] >= DUREE_EXTRAIT
sel, paroles = sel[ok].reset_index(drop=True), [x for x, k in zip(paroles, ok) if k]

rang = sel.groupby("client_id").cumcount()                       # plafond de clips par locuteur
sel, paroles = sel[rang < CLIPS_PAR_LOCUTEUR].reset_index(drop=True), [x for x, r in zip(paroles, rang) if r < CLIPS_PAR_LOCUTEUR]
n_clips = sel["client_id"].map(sel["client_id"].value_counts())
sel, paroles = sel[n_clips >= CLIPS_MIN].reset_index(drop=True), [x for x, n in zip(paroles, n_clips) if n >= CLIPS_MIN]

codes = {cid: f"L{i:04d}" for i, cid in enumerate(sorted(sel["client_id"].unique()))}
sel["locuteur"] = sel["client_id"].map(codes)
sel["f0_locuteur"] = sel["client_id"].map(f0_med).round(1)
assert not set(sel["client_id"]) & EXCLUS, "FUITE détectée !"

# Minimisation : on ne garde ni âge, ni genre, ni accent, ni texte des phrases
sel[["locuteur", "client_id", "clip", "duree_parole", "f0_locuteur"]].to_csv(f"{DOSSIER_DRIVE}/selection_cv27.csv", index=False)

# Audio choisi : un seul zip (beaucoup plus rapide à écrire sur Drive que des milliers de fichiers)
import zipfile
with zipfile.ZipFile(f"{TRAVAIL}/cv27_selection_audio.zip", "w", zipfile.ZIP_STORED) as z:
    for fichier, loc in zip(sel["fichier"], sel["locuteur"]):
        z.write(fichier, f"{loc}/{os.path.basename(fichier)}")
shutil.copy(f"{TRAVAIL}/cv27_selection_audio.zip", f"{DOSSIER_DRIVE}/cv27_selection_audio.zip")

f0s = sel.drop_duplicates("locuteur")["f0_locuteur"]
print(f"Locuteurs : {sel['locuteur'].nunique()} | clips : {len(sel)} | "
      f"clips par locuteur : {sel.groupby('locuteur').size().min()} à {sel.groupby('locuteur').size().max()}")
print(f"Durée de parole par clip : médiane {sel['duree_parole'].median():.1f} s")
print(f"F0 des locuteurs : médiane {f0s.median():.0f} Hz | graves {int((f0s < SEUIL_F0_AIGU).sum())} | aigus {int((f0s >= SEUIL_F0_AIGU).sum())}")
print(f"Zip audio sur Drive : {os.path.getsize(f'{DOSSIER_DRIVE}/cv27_selection_audio.zip')/1e6:.0f} Mo")

In [ ]:
# 11) Empreintes ECAPA sur GPU : 6 versions par clip (extraits de 3 s)
from speechbrain.inference.speaker import EncoderClassifier
appareil = "cuda" if torch.cuda.is_available() else "cpu"
modele = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                        savedir="/content/modeles/ecapa", run_opts={"device": appareil})
modele.eval()

L = int(DUREE_EXTRAIT * 16000)
VERSIONS = ["propre_1", "propre_2", "ambiance_20", "ambiance_10", "brouhaha_20", "brouhaha_10"]
rng_aug = np.random.default_rng(GRAINE + 1)
loc_num = sel["locuteur"].str[1:].astype(int).to_numpy()

def extrait(x):
    debut = rng_aug.integers(0, len(x) - L + 1)
    return x[debut: debut + L]

def bruit_rose():
    # Bruit "d'ambiance" : bruit blanc filtré en 1/f (plus d'énergie dans les graves)
    spectre = np.fft.rfft(rng_aug.standard_normal(L))
    f = np.arange(len(spectre)); f[0] = 1
    return np.fft.irfft(spectre / np.sqrt(f), n=L)

def brouhaha(i_cible):
    # Brouhaha : 5 voix d'AUTRES locuteurs de l'entraînement mélangées
    autres = []
    while len(autres) < 5:
        j = int(rng_aug.integers(len(paroles)))
        if loc_num[j] != loc_num[i_cible]:
            autres.append(extrait(paroles[j]))
    return np.sum(autres, axis=0)

def melanger(signal, bruit, snr_db):
    p_s, p_b = np.mean(signal ** 2) + 1e-10, np.mean(bruit ** 2) + 1e-10
    y = signal + bruit * np.sqrt(p_s / (p_b * 10 ** (snr_db / 10)))
    return y / max(1.0, np.abs(y).max())

def versions_du_clip(i):
    x = paroles[i]
    return [extrait(x), extrait(x),
            melanger(extrait(x), bruit_rose(), 20), melanger(extrait(x), bruit_rose(), 10),
            melanger(extrait(x), brouhaha(i), 20), melanger(extrait(x), brouhaha(i), 10)]

@torch.no_grad()
def empreintes(lot):
    x = torch.from_numpy(np.stack(lot).astype(np.float32)).to(appareil)
    return modele.encode_batch(x, torch.ones(len(lot), device=appareil)).squeeze(1).cpu().numpy()

emb, lot, t0 = [], [], time.time()
for i in range(len(paroles)):
    lot += versions_du_clip(i)
    if len(lot) >= 240 or i == len(paroles) - 1:
        emb.append(empreintes(lot)); lot = []
    if i % 2000 == 0:
        print(f"  clip {i}/{len(paroles)}  ({time.time()-t0:.0f} s)")
emb = np.concatenate(emb).astype(np.float32)

n = len(paroles)
np.savez_compressed(f"{DOSSIER_DRIVE}/emb_cv27_train.npz",
                    emb=emb,                                          # (n*6, 192), brutes (moyenne soustraite plus tard, sur le PC)
                    locuteur=np.repeat(loc_num, len(VERSIONS)),
                    clip=np.repeat(sel["clip"].to_numpy(), len(VERSIONS)),
                    version=np.tile(np.arange(len(VERSIONS)), n),
                    noms_versions=np.array(VERSIONS))

# Contrôle de cohérence Colab <-> PC : même signal synthétique, on comparera l'empreinte sur ton PC
g = np.random.default_rng(0)
t = np.arange(L) / 16000
signal_controle = (0.05 * g.standard_normal(L) + 0.3 * np.sin(2 * np.pi * 150 * t)).astype(np.float32)
np.save(f"{DOSSIER_DRIVE}/controle_ecapa.npy", empreintes([signal_controle])[0])

print(f"\nEmpreintes : {emb.shape} en {time.time()-t0:.0f} s | NaN : {int(np.isnan(emb).sum())}")
print(f"Fichier : {os.path.getsize(f'{DOSSIER_DRIVE}/emb_cv27_train.npz')/1e6:.0f} Mo")

In [ ]:
# 12) Bilan et ménage (minimisation : on supprime l'archive de 28 Go)
for f in [ARCHIVE, ARCHIVE + ".ok"]:
    if os.path.exists(f): os.remove(f)
print("Fichiers dans Mon Drive/voiceprint :")
for f in sorted(os.listdir(DOSSIER_DRIVE)):
    print(f"  {os.path.getsize(os.path.join(DOSSIER_DRIVE, f))/1e6:8.1f} Mo  {f}")
print("\n➡️  Renvoie-moi la sortie des cellules 7, 9, 10, 11 et 12.")